* Primeiramente vamos fazer uma inspeção para saber mais detalhes dos datasets que vamos usar no projeto.

In [1]:
import glob
import os

import pandas as pd

arquivos_csv = glob.glob("/Users/Lased/Documents/RotaCerta/data/raw/*.csv")

def checagem_dfs(arquivos_csv):
    info_arquivos = {}

    for arquivo in arquivos_csv:
        df = pd.read_csv(arquivo)
        linhas, colunas = df.shape
        info_arquivos[arquivo] = (linhas, colunas)

    return info_arquivos

infos = checagem_dfs(arquivos_csv)
for arquivo, (linhas, colunas) in infos.items():
    nome_arquivo = os.path.basename(arquivo)
    print(f"Arquivo: {nome_arquivo} - Linhas: {linhas}, colunas: {colunas}")

Arquivo: avaliacoes.csv - Linhas: 25000, colunas: 4
Arquivo: clientes.csv - Linhas: 8000, colunas: 5
Arquivo: entregas.csv - Linhas: 25000, colunas: 6
Arquivo: pedidos.csv - Linhas: 25000, colunas: 10


* Vamos fazer uma inspeção para entender mais sobre os dados que vamos trabalhar nesse projeto.

In [ ]:
import warnings

import numpy as np

warnings.filterwarnings('ignore')

def inspecao(df, nome_dataset="Dataset"):

    """Realiza um profiling exploratório inicial de um DataFrame.

    Imprime um relatório com shape, tipos de dados, valores nulos,
    checagem de colunas de data, duplicatas, estatísticas descritivas
    das colunas numéricas, cardinalidade das colunas categóricas/booleanas
    e a janela temporal das colunas de data.

    Args:
        df (pd.DataFrame): DataFrame a ser inspecionado.
        nome_dataset (str, opcional): Nome de exibição do dataset no
            cabeçalho do relatório. Padrão: "Dataset".

    Returns:
        None: a função apenas imprime o relatório no console/notebook,
        não retorna nenhum valor.
    """
    
    print("-=" * 30)
    print(f"INSPEÇÃO: {nome_dataset.upper()}")
    
    # 1. Shape:
    print(f"\n Shape do dataset: {df.shape[0]} linhas e {df.shape[1]} colunas")

    # 2. Tipos de Dados:
    print("\n Tipo de dado de cada coluna:")
    print(df.dtypes)

    # 3. Valores nulos(quantidade e porcentagem):
    nulos = df.isnull().sum()
    if nulos.sum() > 0:
        df_nulos = pd.DataFrame({'Qtd_Nulos': nulos, 'Porcentagem': (nulos / len(df)) * 100})
        print("\n Valores nulos encontrados no dataset:")
        print(df_nulos)
    else:
        print("\n Nenhum valor nulo encontrado.")

    # 4. Checando as colunas de data para saber se são ou não objetos datetime:
    print("\n Checagem das colunas de data:")
    # Procura qualquer coluna que tenha a palavra "data" no nome
    colunas_de_data = [col for col in df.columns if 'data' in col.lower()]

    if colunas_de_data:
        for col in colunas_de_data:
            # Checa o tipo e já imprime o resultado
            is_datetime = pd.api.types.is_datetime64_any_dtype(df[col]) 
            print(f"-> A coluna {col} está no formato datetime? {is_datetime}") # True = a coluna é datetime | False = a coluna não é datetime
    else:
        print("-> Nenhuma coluna com a palavra 'data' foi encontrada.")

    # 5. Checagem de duplicatas:
    print(f"\n Linhas totalmente duplicadas: {df.duplicated().sum()}")

    # 6. Ranges e Sanidade Numérica:
    colunas_numericas = df.select_dtypes(include=np.number).columns
    if len(colunas_numericas) > 0:
        print("\n Estatísticas Descritivas (Númericas):")
        print(df[colunas_numericas].describe().round(2))

    # 7. Cardinalidade de Categóricas:
    colunas_categoricas = df.select_dtypes(include=['object', 'category', 'bool']).columns
    if len(colunas_categoricas) > 0:
        print("\n Cardinalidade(Categóricas/Booleanas):")
        for col in colunas_categoricas:
            valores_unicos = df[col].nunique()
            print(f" -> {col}: {valores_unicos} categorias exclusivas")
            # Mostrar o top 3 de valores:
            top3 = df[col].value_counts(normalize=True).head(3) * 100
            print(f" Top categorias: {top3.round(2).to_dict()}")

    # 8. Limites de Tempo:
    if colunas_de_data:
        print("\n Janela de tempo(Datas):")
        for col in colunas_de_data:
            if pd.api.types.is_datetime64_any_dtype(df[col]):
                print(f" -> {col}: de {df[col].min()} até {df[col].max()}")
            else:
                print(f"-> A coluna '{col}' precisa ser convertida para datetime!")
    

    print("-=" * 30)

* Usando a função inspecao() nos datasets:

In [3]:
clientes = pd.read_csv("C:/Users/Lased/Documents/RotaCerta/data/raw/clientes.csv")
inspecao(clientes, "Clientes")

-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=
INSPEÇÃO: CLIENTES

 Shape do dataset: 8000 linhas e 5 colunas

 Tipo de dado de cada coluna:
cliente_id                int64
regiao                      str
renda_estimada          float64
canal_aquisicao             str
data_primeira_compra        str
dtype: object

 Valores nulos encontrados no dataset:
                      Qtd_Nulos  Porcentagem
cliente_id                    0          0.0
regiao                        0          0.0
renda_estimada              120          1.5
canal_aquisicao               0          0.0
data_primeira_compra          0          0.0

 Checagem das colunas de data:
-> A coluna data_primeira_compra está no formato datetime? False

 Linhas totalmente duplicadas: 0

 Estatísticas Descritivas (Númericas):
       cliente_id  renda_estimada
count     8000.00         7880.00
mean      4000.50         3220.40
std       2309.55         1706.13
min          1.00          338.31
25%       2000.75   

In [4]:
pedidos = pd.read_csv(r"C:\Users\Lased\Documents\RotaCerta\data\raw\pedidos.csv")
inspecao(pedidos, "Pedidos")

-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=
INSPEÇÃO: PEDIDOS

 Shape do dataset: 25000 linhas e 10 colunas

 Tipo de dado de cada coluna:
pedido_id                int64
cliente_id               int64
data_pedido                str
categoria_produto          str
preco                  float64
valor_frete            float64
distancia_km           float64
prazo_estimado_dias      int64
prazo_real_dias          int64
estacao_do_ano             str
dtype: object

 Nenhum valor nulo encontrado.

 Checagem das colunas de data:
-> A coluna data_pedido está no formato datetime? False

 Linhas totalmente duplicadas: 0

 Estatísticas Descritivas (Númericas):
       pedido_id  cliente_id     preco  valor_frete  distancia_km  \
count   25000.00    25000.00  25000.00     25000.00      25000.00   
mean    12500.50     4007.99    144.35        15.74        283.85   
std      7217.02     2307.54    188.30        10.04        135.14   
min         1.00        1.00      7.26         5.0

In [5]:
entregas = pd.read_csv(r"C:\Users\Lased\Documents\RotaCerta\data\raw\entregas.csv")
inspecao(entregas, "Entregas")

-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=


INSPEÇÃO: ENTREGAS

 Shape do dataset: 25000 linhas e 6 colunas

 Tipo de dado de cada coluna:
pedido_id            int64
transportadora         str
regiao_origem          str
regiao_destino         str
ocorrencia_atraso     bool
motivo_atraso          str
dtype: object

 Valores nulos encontrados no dataset:
                   Qtd_Nulos  Porcentagem
pedido_id                  0        0.000
transportadora             0        0.000
regiao_origem              0        0.000
regiao_destino             0        0.000
ocorrencia_atraso          0        0.000
motivo_atraso          20827       83.308

 Checagem das colunas de data:
-> Nenhuma coluna com a palavra 'data' foi encontrada.

 Linhas totalmente duplicadas: 0

 Estatísticas Descritivas (Númericas):
       pedido_id
count   25000.00
mean    12500.50
std      7217.02
min         1.00
25%      6250.75
50%     12500.50
75%     18750.25
max     25000.00

 Cardinalidade(Categóricas/Booleanas):
 -> transportadora: 3 categorias exclusi

In [6]:
avaliacoes = pd.read_csv(r"C:\Users\Lased\Documents\RotaCerta\data\raw\avaliacoes.csv")
inspecao(avaliacoes, "Avaliações")

-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=
INSPEÇÃO: AVALIAÇÕES

 Shape do dataset: 25000 linhas e 4 colunas

 Tipo de dado de cada coluna:
pedido_id                   int64
nota                      float64
tempo_ate_avaliar_dias      int64
comprou_novamente_90d        bool
dtype: object

 Valores nulos encontrados no dataset:
                        Qtd_Nulos  Porcentagem
pedido_id                       0          0.0
nota                          250          1.0
tempo_ate_avaliar_dias          0          0.0
comprou_novamente_90d           0          0.0

 Checagem das colunas de data:
-> Nenhuma coluna com a palavra 'data' foi encontrada.

 Linhas totalmente duplicadas: 0

 Estatísticas Descritivas (Númericas):
       pedido_id      nota  tempo_ate_avaliar_dias
count   25000.00  24750.00                25000.00
mean    12500.50      3.94                    5.97
std      7217.02      0.88                    4.21
min         1.00      1.00                    1.00
2